# Mini Project 1 — Analysis Notebook

**Your name:**  Auli Badoni

**Dataset:**  Billboard Top 100

**Date:**  6th May 2026

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [91]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


In [92]:
# Additional imports needed for this analysis
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["numpy", "requests", "pillow"]:
    try:
        __import__("PIL" if pkg == "pillow" else pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import requests
from PIL import Image
from io import BytesIO
print("All libraries loaded successfully")

All libraries loaded successfully


---

## Section 1 — Overview
# Billboard Hot 100 Analysis | 2023–2026

**Dataset:** Weekly Billboard Hot 100 chart data collected using the `billboard.py` 
Python package, which scrapes chart data from Billboard.com. Covers 174 weekly 
charts from January 7, 2023 to May 2, 2026 — 17,400 rows, 100 songs per week.

**Why this dataset:** Billboard chart data offers a structured way to study how 
audiences collectively respond to music over time, connecting to HCD themes of 
attention, popularity, and sustained engagement with creative products.

**Three analytical questions:**

1. Which songs and artists had the strongest chart presence between 2023 and 2026, 
   based on total weekly appearances, Top 10 appearances, and 1 placements?
2. How do songs typically move through the Billboard Hot 100: do they slowly climb, 
   debut high, remain stable, or drop quickly after peaking?
3. Has chart behavior changed across 2023–2026 in terms of new entries, average 
   weeks on chart, and rank movement volatility?

**What a practitioner would do with these findings:** Music label A&R teams and 
streaming platform curators could use these trajectory patterns to identify 
slow-burn hits early and adjust playlist promotion timing accordingly.

**Data provenance note:** Data was scraped via `billboard.py` from Billboard.com. 
Gaps may exist where Billboard's site structure changed or rate limits were hit. 
Top-ranked songs (ranks 1–4) frequently have missing image URLs due to scraping 
behavior — handled in cleaning.

---

## Section 2 — Data Profile

**Shape:** 17,400 rows × 16 columns (plus 2 derived columns added during cleaning)

**What each column represents:**

- `chart_date` — the Saturday the chart was published (weekly)
- `rank` — song's position that week (1 = best, 100 = worst)
- `song` — song title
- `artist` — full credited artist string including features
- `last_week` — rank from the previous week (null for new entries & re-entries)
- `peak_rank` — best rank the song has ever achieved (cumulative, set by Billboard)
- `weeks_on_board` — total weeks the song has appeared on the chart (cumulative)
- `is_new` — True if this is the song's first week on the chart
- `image_url` — Billboard CDN URL for the song's cover art (frequently null for top ranks)
- `year` — extracted from chart_date
- `month` — extracted from chart_date
- `decade` — decade of chart_date (all 2020s in this dataset)
- `is_top_10` — True if rank ≤ 10
- `is_number_one` — True if rank == 1
- `rank_change` — how many positions the song moved vs last week (positive = up)
- `movement_type` — categorical: new, up, down, same, re-entry

**Derived columns added during cleaning:**
- `primary_artist` — first-billed artist extracted from the full artist string
- `featured_artists` — list of all non-primary credited artists

**Data quality issues found:**
- `last_week` and `rank_change` are null for 2,788 rows — expected behavior 
  for new entries (2,044) and re-entries (744), not treated as missing data
- `image_url` is null for 696 rows — Billboard does not consistently serve 
  images for top-ranked songs via scraping
- `artist` column contains inconsistent collaboration formatting 
  (& vs Featuring vs X vs ,) — resolved via primary_artist extraction

**Columns this analysis focuses on:**
- `rank`, `chart_date`, `primary_artist`, `song` — core identifiers and metrics
- `is_top_10`, `is_number_one` — for Q1 dominance analysis
- `rank_change`, `movement_type` — for Q2 trajectory and Q3 volatility
- `is_new`, `weeks_on_board` — for Q3 trend analysis over time

In [93]:
# Load your dataset
df = pd.read_csv('data/billboard_hot100_2023_2026.csv')

print(df.shape)
df.head()

(17400, 16)


,chart_date,rank,song,artist,last_week,peak_rank,weeks_on_board,is_new,image_url,year,month,decade,is_top_10,is_number_one,rank_change,movement_type
0,2023-01-07,1,All I Want For Christmas Is You,Mariah Carey,1.0,1,58,False,NaN,2023,1,2020,True,True,0.0,same
1,2023-01-07,2,Rockin' Around The Christmas Tree,Brenda Lee,2.0,2,52,False,NaN,2023,1,2020,True,False,0.0,same
2,2023-01-07,3,Jingle Bell Rock,Bobby Helms,3.0,3,49,False,NaN,2023,1,2020,True,False,0.0,same
3,2023-01-07,4,Last Christmas,Wham!,5.0,4,31,False,NaN,2023,1,2020,True,False,1.0,up
4,2023-01-07,5,A Holly Jolly Christmas,Burl Ives,4.0,4,32,False,https://charts-static.billboard.com/img/1998/0...,2023,1,2020,True,False,-1.0,down


In [94]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17400 entries, 0 to 17399
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   chart_date      17400 non-null  str    
 1   rank            17400 non-null  int64  
 2   song            17400 non-null  str    
 3   artist          17400 non-null  str    
 4   last_week       14612 non-null  float64
 5   peak_rank       17400 non-null  int64  
 6   weeks_on_board  17400 non-null  int64  
 7   is_new          17400 non-null  bool   
 8   image_url       16704 non-null  str    
 9   year            17400 non-null  int64  
 10  month           17400 non-null  int64  
 11  decade          17400 non-null  int64  
 12  is_top_10       17400 non-null  bool   
 13  is_number_one   17400 non-null  bool   
 14  rank_change     14612 non-null  float64
 15  movement_type   17400 non-null  str    
dtypes: bool(3), float64(2), int64(6), str(5)
memory usage: 1.8 MB


In [95]:
# Summary statistics for numeric columns
df.describe()

,rank,last_week,peak_rank,weeks_on_board,year,month,decade,rank_change
count,17400.0000,14612.000000,17400.000000,17400.000000,17400.000000,17400.000000,17400.0,14612.000000
mean,50.5000,45.918765,32.615000,13.720575,2024.206897,6.143678,2020.0,-2.272310
std,28.8669,27.456324,27.359857,13.656719,0.984248,3.488511,0.0,12.507993
min,1.0000,1.000000,1.000000,1.000000,2023.000000,1.000000,2020.0,-73.000000
25%,25.7500,22.000000,8.000000,4.000000,2023.000000,3.000000,2020.0,-7.000000
50%,50.5000,45.000000,26.000000,10.000000,2024.000000,6.000000,2020.0,-1.000000
75%,75.2500,69.000000,53.000000,19.000000,2025.000000,9.000000,2020.0,3.000000
max,100.0000,100.000000,100.000000,112.000000,2026.000000,12.000000,2020.0,79.000000


In [96]:
# Plotly/Kaleido: fetch Chromium for static image export (pipe y to the prompt)
!echo y | plotly_get_chrome



Plotly will install a copy of Google Chrome to be used for generating static images of plots.
Chrome will be installed at: None
Do you want to proceed? [y/n] Installing Chrome for Plotly...
Chrome installed successfully.
The Chrome executable is now located at: /Users/auli/Library/Application Support/choreographer/deps/chrome-mac-arm64/Google Chrome for Testing.app/Contents/MacOS/Google Chrome for Testing


**My data profile notes:**

When I first loaded the dataset I noticed it was largely clean out of the box — 
no corrupt rows, no duplicate chart weeks, and all core fields like rank, song, 
and artist were fully populated. The 2,788 null values in last_week and 
rank_change looked alarming at first but turned out to be expected — they all 
correspond to new entries and re-entries, which by definition have no previous 
week to reference.

The artist column immediately stood out as inconsistent — the same collaboration 
can appear as "Drake & 21 Savage", "Drake Featuring 21 Savage", or 
"Metro Boomin, The Weeknd & 21 Savage" depending on the song. This will need 
cleaning before any artist-level aggregation.

The weeks_on_board column is cumulative — it reflects a song's total chart 
history, not just weeks in this dataset. This means some songs like Mariah 
Carey's Christmas songs appear with 50+ weeks even though they only charted 
for a few weeks within my date range. I will need to handle this carefully 
for Q1 and Q3.

One question this raises: how do I count an artist's chart presence fairly 
when one artist releases 30 songs at once (Morgan Wallen) vs another who 
has 3 songs that each dominate for months (Taylor Swift)?

### Fix Data Types
Convert chart_date to datetime and ensure numeric columns are correct type.

In [97]:
### Fix Data Types

# Convert chart_date from string to datetime
df['chart_date'] = pd.to_datetime(df['chart_date'])

# Convert numeric columns (coerce errors handles nulls gracefully)
numeric_cols = ['rank', 'last_week', 'peak_rank', 'weeks_on_board', 'rank_change']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Confirm types
print(df.dtypes)

chart_date        datetime64[us]
rank                       int64
song                         str
artist                       str
last_week                float64
peak_rank                  int64
weeks_on_board             int64
is_new                      bool
image_url                    str
year                       int64
month                      int64
decade                     int64
is_top_10                   bool
is_number_one               bool
rank_change              float64
movement_type                str
dtype: object


### Handle Missing Values
Inspect nulls — note that last_week and rank_change being null 
is expected for new entries and re-entries.

In [98]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())
print()

# Confirm nulls are only from new entries and re-entries
print("Null last_week breakdown by movement_type:")
print(df[df['last_week'].isna()]['movement_type'].value_counts())
print()

# Drop only rows where critical fields are null
before = df.shape[0]
df = df.dropna(subset=['rank', 'song', 'artist'])
after = df.shape[0]
print(f"Rows before: {before} | Rows after: {after} | Dropped: {before - after}")

Missing values per column:
chart_date           0
rank                 0
song                 0
artist               0
last_week         2788
peak_rank            0
weeks_on_board       0
is_new               0
image_url          696
year                 0
month                0
decade               0
is_top_10            0
is_number_one        0
rank_change       2788
movement_type        0
dtype: int64

Null last_week breakdown by movement_type:
movement_type
new         2044
re-entry     744
Name: count, dtype: int64

Rows before: 17400 | Rows after: 17400 | Dropped: 0


###  Extract Primary Artist and Featured Artists
Many entries include collaborations — extract the first-billed 
artist as primary_artist for clean grouping.


In [99]:
import re

def extract_primary_artist(artist):
    primary = re.split(r' & | Featuring | X | x |, ', artist)[0]
    return primary.strip()

def extract_featured_artists(artist):
    parts = re.split(r' & | Featuring | X | x |, ', artist)
    features = [p.strip() for p in parts[1:] if p.strip()]
    return features if features else None

df['primary_artist'] = df['artist'].apply(extract_primary_artist)
df['featured_artists'] = df['artist'].apply(extract_featured_artists)

# Create unique song key
df['song_key'] = df['song'] + ' — ' + df['primary_artist']

# Sanity check
print("Sample with features:")
print(df[df['featured_artists'].notna()][['artist', 'primary_artist', 'featured_artists']].head(5).to_string())
print(f"\nNew shape: {df.shape}")

Sample with features:
                                                                     artist                      primary_artist                     featured_artists
9                                                    Sam Smith & Kim Petras                           Sam Smith                         [Kim Petras]
15  Bing Crosby With Ken Darby Singers & John Scott Trotter & His Orchestra  Bing Crosby With Ken Darby Singers  [John Scott Trotter, His Orchestra]
18                                                David Guetta & Bebe Rexha                        David Guetta                         [Bebe Rexha]
19              Frank Sinatra With The Orchestra & Chorus Of Gordon Jenkins    Frank Sinatra With The Orchestra           [Chorus Of Gordon Jenkins]
20                                                        Drake & 21 Savage                               Drake                          [21 Savage]

New shape: (17400, 19)


###  Build Image Lookup + Final Sanity Check (Pulling Images for the graphs)
Build an artist and song image lookup dictionary for use in charts,
then confirm the dataset is clean and ready for analysis.

In [100]:
# --- IMAGE LOOKUP: song -> image_url ---
image_lookup = (df[df['image_url'].notna()]
                .groupby('song_key')['image_url']
                .first()
                .to_dict())

# --- IMAGE LOOKUP: artist -> image_url ---
artist_image_lookup = (df[df['image_url'].notna()]
                       .sort_values('chart_date', ascending=False)
                       .drop_duplicates(subset='primary_artist')
                       .set_index('primary_artist')['image_url']
                       .to_dict())

print(f"Songs with image URLs: {len(image_lookup)}")
print(f"Artists with image URLs: {len(artist_image_lookup)}")
print()

# --- FINAL SANITY CHECK ---
print("=== FINAL DATASET SUMMARY ===")
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"Date range: {df['chart_date'].min().date()} → {df['chart_date'].max().date()}")
print(f"Unique songs: {df['song'].nunique()}")
print(f"Unique primary artists: {df['primary_artist'].nunique()}")
print(f"Unique chart weeks: {df['chart_date'].nunique()}")
print()
print("Movement type breakdown:")
print(df['movement_type'].value_counts())
print()
print("Top 5 songs by weeks on chart:")
print(df.groupby('song_key')['weeks_on_board'].max()
      .sort_values(ascending=False).head(5))

Songs with image URLs: 2228
Artists with image URLs: 506

=== FINAL DATASET SUMMARY ===
Total rows: 17400
Total columns: 19
Date range: 2023-01-07 → 2026-05-02
Unique songs: 2181
Unique primary artists: 506
Unique chart weeks: 174

Movement type breakdown:
movement_type
down        7882
up          5493
new         2044
same        1237
re-entry     744
Name: count, dtype: int64

Top 5 songs by weeks on chart:
song_key
Lose Control — Teddy Swims                        112
Beautiful Things — Benson Boone                    89
All I Want For Christmas Is You — Mariah Carey     79
A Bar Song (Tipsy) — Shaboozey                     77
Wildflower — Billie Eilish                         72
Name: weeks_on_board, dtype: int64


In [101]:
# --- Helpers for chart interactivity: text hover + on-canvas album covers ---
# Plotly's hovertemplate is rendered inside SVG <text>, which can't render <img>
# tags, so hovers stay text-only. For on-canvas covers we pre-fetch each
# Billboard image and embed it as a base64 data URI — the Billboard CDN blocks
# cross-origin loading inside SVG <image> nodes, so passing the raw URL to
# layout.images doesn't render reliably in the browser.

import base64, requests

def build_hover_text(song, artist, stats_lines):
    """Text-only hovertemplate string (safe inside Plotly's SVG hover label)."""
    stats = "<br>".join(stats_lines)
    return f"<b>{song}</b><br>{artist}<br>{stats}<extra></extra>"

HOVER_LABEL = dict(
    bgcolor="#fdf0f5",
    bordercolor="#e8c8e0",
    font=dict(color="#2d1a2e", size=12),
    align="left",
)

_b64_cache = {}

def fetch_image_as_b64(url, timeout=5):
    """Download an image and return a data: URI; cached per URL. None on failure."""
    if not url:
        return None
    if url in _b64_cache:
        return _b64_cache[url]
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        mime = r.headers.get('Content-Type', 'image/jpeg').split(';')[0]
        data_uri = f"data:{mime};base64,{base64.b64encode(r.content).decode()}"
    except Exception:
        data_uri = None
    _b64_cache[url] = data_uri
    return data_uri

def cover_layout_image(image_url_or_b64, x, y, sizex, sizey,
                       xref="x", yref="y",
                       xanchor="center", yanchor="middle",
                       layer="above"):
    """Build a layout.images entry for an album cover anchored to chart coords."""
    if not image_url_or_b64:
        return None
    return dict(
        source=image_url_or_b64,
        x=x, y=y,
        sizex=sizex, sizey=sizey,
        xref=xref, yref=yref,
        xanchor=xanchor, yanchor=yanchor,
        layer=layer,
        sizing="contain",
    )

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** *Which songs and artists had the strongest chart presence between 2023 and 2026, 
   based on total weekly appearances, Top 10 appearances, and 1 placements?*

In [102]:
## Section 4 — Analysis
### Chart 1: Longest-Running Top 10 Songs (2023–2026)

# --- TOP 10 MOST CHARTED SONGS: Overall + Per Year ---

# --- BUILD OVERALL TOP 10 ---
overall = df.groupby(['song', 'primary_artist']).agg(
    top10_weeks   = ('is_top_10', 'sum'),
    number1_weeks = ('is_number_one', 'sum'),
    total_weeks   = ('rank', 'count'),
    peak_rank     = ('rank', 'min')
).reset_index()

overall = overall.sort_values('top10_weeks', ascending=True).tail(10)
overall['label'] = '<b>' + overall['song'] + '</b><br>' + overall['primary_artist']
overall['song_key'] = overall['song'] + ' — ' + overall['primary_artist']
overall['image_url'] = overall['song_key'].map(image_lookup)
overall['hover_tpl'] = overall.apply(
    lambda r: build_hover_text(
        r['song'], r['primary_artist'],
        [f"Peak Rank: #{r['peak_rank']}",
         f"Weeks in Top 10: {r['top10_weeks']}",
         f"Weeks at #1: {r['number1_weeks']}"],
    ), axis=1,
)

# Pre-fetch each album cover as a base64 data URI (Billboard CDN blocks
# cross-origin SVG <image> loads, so raw URLs don't render reliably).
print("Fetching album covers...")
overall['image_b64'] = overall['image_url'].map(fetch_image_as_b64)
print(f"  loaded {overall['image_b64'].notna().sum()} / {len(overall)} covers")

# Build custom row of [cover | song + artist] to the LEFT of each bar.
# Both cover and label are RIGHT-anchored so no label can ever extend into
# the plot area, regardless of title length.
COVER_X_RIGHT = -0.23   # cover's right edge, in paper coords
COVER_W_PAPER = 0.06
LABEL_X       = -0.01   # label's right edge, just left of plot

cover_images = []
row_label_annotations = []
for label_html, b64 in zip(overall['label'], overall['image_b64']):
    cover = cover_layout_image(
        b64,
        x=COVER_X_RIGHT, y=label_html,
        sizex=COVER_W_PAPER, sizey=0.85,
        xref='paper', yref='y',
        xanchor='right', yanchor='middle',
    )
    if cover:
        cover_images.append(cover)
    row_label_annotations.append(dict(
        x=LABEL_X, y=label_html,
        xref='paper', yref='y',
        text=label_html,
        showarrow=False,
        xanchor='right', yanchor='middle',
        font=dict(size=11, color='#2d1a2e'),
        align='right',
    ))

# --- PLOT ---
fig = go.Figure()

fig.add_trace(go.Bar(
    y=overall['label'],
    x=overall['top10_weeks'],
    orientation='h',
    name='Weeks in Top 10',
    marker_color='#7b2d8b',
    hovertemplate=overall['hover_tpl'],
    hoverlabel=HOVER_LABEL,
))

fig.add_trace(go.Bar(
    y=overall['label'],
    x=overall['number1_weeks'],
    orientation='h',
    name='Weeks at #1',
    marker_color='#e91e8c',
    hovertemplate=overall['hover_tpl'],
    hoverlabel=HOVER_LABEL,
))

fig.update_layout(
    title=dict(
        text='<b>Billboard Hot 100</b><br>'
             '<sup>Top 10 Most Charted Songs (2023–2026)</sup>',
        font=dict(size=20, color='#2d1a2e'),
        x=0.5
    ),
    barmode='overlay',
    xaxis=dict(
        title='Weeks in Top 10',
        gridcolor='#e8c8e0',
        tickfont=dict(color='#666666'),
    ),
    yaxis=dict(
        showticklabels=False,
    ),
    annotations=row_label_annotations,
    plot_bgcolor='#fdf0f5',
    paper_bgcolor='#f5eaf5',
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.12,
        xanchor='center',
        x=0.5,
        font=dict(size=12, color='#2d1a2e'),
        bgcolor='rgba(253,240,245,0.8)',
        bordercolor='#e8c8e0',
        borderwidth=1
    ),

    height=600,
    margin=dict(l=280, r=100, t=100, b=60),
    images=cover_images,
)

fig.show()
fig.write_image('chart1_songs_top10.png', width=1400, height=700, scale=2)
print("Saved as chart1_songs_top10.png")

Fetching album covers...
  loaded 10 / 10 covers


Saved as chart1_songs_top10.png


**Interpretation:**

The first thing this chart makes clear is that longevity and dominance are not the same metric. Lose Control by Teddy Swims leads on weeks in the Top 10 (80) but only spent 1 week at #1, while A Bar Song (Tipsy) by Shaboozey has fewer Top-10 weeks (66) but the most weeks at #1 of any song in the dataset (19). Several songs — Espresso (Sabrina Carpenter) and Birds Of A Feather (Billie Eilish), both 33 Top-10 weeks and 0 weeks at #1 — show that a song can be everywhere on the chart for months without ever reaching the top spot.

I expected a few songs to dominate, which they do, but I did not expect the “most charted” and “most #1” leaders to be different songs — that gap is the clearest finding here. The main limitation is that this view is song-level, and `primary_artist` collapses collaborations to the first-billed name, so it does not answer “which artist had the most total presence across many singles.” As a follow-up I would do an artist-level rollup (count of distinct songs plus sum of chart weeks per `primary_artist`) so an artist like Morgan Wallen, who charts many songs at once, is comparable to Taylor Swift, whose individual songs each chart for longer.

**Question 2:** *How do songs typically move through the Billboard Hot 100: do they slowly climb, 
   debut high, remain stable, or drop quickly after peaking?*

In [103]:
# Q2 — STEP 1: classify each song's trajectory shape on the Hot 100
# --- STEP 1: GET TOP 20 MOST CHARTED SONGS ---
# Create unique song key to avoid title conflicts

df['song_key'] = df['song'] + ' — ' + df['primary_artist']

# Count total appearances per song
song_counts = df.groupby('song_key')['rank'].count().reset_index()
song_counts.columns = ['song_key', 'appearance_count']

# Filter to top 20
top20_keys = song_counts.sort_values('appearance_count', ascending=False).head(20)['song_key'].tolist()

# Filter main df to only those songs
df_top20 = df[df['song_key'].isin(top20_keys)].copy()

# Sort by song and date
df_top20 = df_top20.sort_values(['song_key', 'chart_date'])

# Assign week number per song (1, 2, 3...)
df_top20['week_number'] = df_top20.groupby('song_key').cumcount() + 1

# Compute chart score (inverted rank so higher = better)
df_top20['chart_score'] = 101 - df_top20['rank']

# Preview
print(f"Songs in trajectory dataset: {df_top20['song_key'].nunique()}")
print(df_top20[['song_key', 'chart_date', 'rank', 'week_number', 'chart_score']].head(10))

Songs in trajectory dataset: 20
                            song_key chart_date  rank  week_number  \
6835  A Bar Song (Tipsy) — Shaboozey 2024-04-27    36            1   
6926  A Bar Song (Tipsy) — Shaboozey 2024-05-04    27            2   
7002  A Bar Song (Tipsy) — Shaboozey 2024-05-11     3            3   
7104  A Bar Song (Tipsy) — Shaboozey 2024-05-18     5            4   
7203  A Bar Song (Tipsy) — Shaboozey 2024-05-25     4            5   
7303  A Bar Song (Tipsy) — Shaboozey 2024-06-01     4            6   
7403  A Bar Song (Tipsy) — Shaboozey 2024-06-08     4            7   
7503  A Bar Song (Tipsy) — Shaboozey 2024-06-15     4            8   
7603  A Bar Song (Tipsy) — Shaboozey 2024-06-22     4            9   
7702  A Bar Song (Tipsy) — Shaboozey 2024-06-29     3           10   

      chart_score  
6835           65  
6926           74  
7002           98  
7104           96  
7203           97  
7303           97  
7403           97  
7503           97  
7603           97

In [104]:
# Your analysis for Question 2
# --- PICK BEST EXAMPLE OF EACH TRAJECTORY TYPE ---
def _trajectory_for(group):
    peak_idx      = group['chart_score'].idxmax()
    peak_week     = group.loc[peak_idx, 'week_number']
    total_weeks   = group['week_number'].max()
    peak_position = peak_week / total_weeks

    if peak_position <= 0.25:
        return 'Debut High'
    elif peak_position >= 0.60:
        return 'Slow Climber'
    return 'Spike & Drop'

trajectory_by_song = (
    df_top20
    .groupby('song_key', sort=False)
    .apply(_trajectory_for)
)
df_top20['trajectory'] = df_top20['song_key'].map(trajectory_by_song)

print(df_top20['trajectory'].value_counts())
print()
print(df_top20[['song_key', 'trajectory']].drop_duplicates().to_string())

trajectory
Debut High      646
Spike & Drop    473
Slow Climber    175
Name: count, dtype: int64

                                 song_key    trajectory
6835       A Bar Song (Tipsy) — Shaboozey    Debut High
5614      Beautiful Things — Benson Boone    Debut High
7312   Birds Of A Feather — Billie Eilish  Spike & Drop
2148          Cruel Summer — Taylor Swift  Spike & Drop
8602         Die With A Smile — Lady Gaga  Spike & Drop
6806         Espresso — Sabrina Carpenter    Debut High
1343                Fast Car — Luke Combs    Debut High
300                 Flowers — Miley Cyrus    Debut High
6776     Good Luck, Babe! — Chappell Roan  Spike & Drop
7200        I Had Some Help — Post Malone    Debut High
526            Last Night — Morgan Wallen    Debut High
3398           Lose Control — Teddy Swims  Spike & Drop
7100         Not Like Us — Kendrick Lamar    Debut High
11160              Ordinary — Alex Warren  Spike & Drop
7789       Pink Pony Club — Chappell Roan  Slow Climber
73    

In [105]:
# Q2 — STEP 3: Pick best example per trajectory type + plot

# Pick song with most weeks per trajectory type
best_per_type = (
    df_top20.groupby(['song_key', 'trajectory'])['week_number']
    .max()
    .reset_index()
    .sort_values('week_number', ascending=False)
    .drop_duplicates(subset='trajectory')
)
print("Selected songs:")
print(best_per_type[['song_key', 'trajectory', 'week_number']].to_string())

selected_keys = best_per_type['song_key'].tolist()
df_selected = df_top20[df_top20['song_key'].isin(selected_keys)].copy()

color_map = {
    'Debut High'  : '#7b2d8b',
    'Spike & Drop': '#f5a623',
    'Slow Climber': '#e91e8c',
}

fig = go.Figure()

for song_key, group in df_selected.groupby('song_key'):
    trajectory  = group['trajectory'].iloc[0]
    color       = color_map[trajectory]
    short_name  = song_key.split(' — ')[0]
    artist_name = song_key.split(' — ')[1]
    group       = group.sort_values('week_number')

    entry_date = group['chart_date'].min().strftime('%b %Y')
    last_date  = group['chart_date'].max().strftime('%b %Y')
    peak_idx   = group['chart_score'].idxmax()
    peak_score = group.loc[peak_idx, 'chart_score']
    peak_week  = group.loc[peak_idx, 'week_number']
    peak_rank  = 101 - int(peak_score)

    hover_tpl = [
        build_hover_text(
            short_name, f"{artist_name} — {trajectory}",
            [f"Week: {int(row['week_number'])} ({pd.to_datetime(row['chart_date']).strftime('%b %Y')})",
             f"Rank: #{int(row['rank'])}",
             f"Score: {int(row['chart_score'])}"],
        )
        for _, row in group.iterrows()
    ]

    # Main line
    fig.add_trace(go.Scatter(
        x=group['week_number'],
        y=group['chart_score'],
        mode='lines',
        name=f"{short_name}<br><sub>{artist_name} — {trajectory}</sub>",
        line=dict(color=color, width=3),
        hovertemplate=hover_tpl,
        hoverlabel=HOVER_LABEL,
    ))

    # Peak dot
    fig.add_trace(go.Scatter(
        x=[peak_week],
        y=[peak_score],
        mode='markers',
        marker=dict(
            color=color,
            size=12,
            line=dict(color='white', width=2)
        ),
        showlegend=False,
        hovertext=[f"<b>Peak!</b><br>Rank #{peak_rank}<br>Week {int(peak_week)}"],
        hoverinfo='text',
        hoverlabel=dict(bgcolor=color, font=dict(color='white'))
    ))

# Reference lines
fig.add_hline(
    y=91, line_dash='dot', line_color='#aaaaaa', line_width=1,
    annotation_text='Top 10',
    annotation_position='bottom left',
    annotation_font=dict(color='#aaaaaa', size=10)
)
fig.add_hline(
    y=100, line_dash='dot', line_color='#aaaaaa', line_width=1,
    annotation_text='#1',
    annotation_position='bottom left',
    annotation_font=dict(color='#aaaaaa', size=10)
)

fig.update_layout(
    title=dict(
        text='<b>Billboard Hot 100</b><br>'
             '<sup>Three Ways a Song Can Win (2023–2026)</sup>',
        font=dict(size=20, color='#2d1a2e'),
        x=0.5
    ),
    xaxis=dict(
        title='Week Number on Chart',
        gridcolor='#e8c8e0',
        tickfont=dict(color='#666666'),
        zeroline=False
    ),
    yaxis=dict(
        title='Chart Score (101 − Rank)<br>Higher = Better',
        gridcolor='#e8c8e0',
        tickfont=dict(color='#666666'),
        range=[0, 115]
    ),
    plot_bgcolor='#fdf0f5',
    paper_bgcolor='#f5eaf5',
    legend=dict(
        orientation='v',
        yanchor='middle',
        y=0.5,
        xanchor='left',
        x=1.02,
        font=dict(size=11, color='#2d1a2e'),
        bgcolor='rgba(253,240,245,0.8)',
        bordercolor='#e8c8e0',
        borderwidth=1
    ),
    height=600,
    margin=dict(l=80, r=200, t=100, b=60),
    hovermode='closest',
)

fig.show()
fig.write_image('chart2_song_trajectories.png', width=1500, height=700, scale=2)
print("Saved as chart2_song_trajectories.png")

Selected songs:
                           song_key    trajectory  week_number
11       Lose Control — Teddy Swims  Spike & Drop          112
1   Beautiful Things — Benson Boone    Debut High           89
14   Pink Pony Club — Chappell Roan  Slow Climber           68


Saved as chart2_song_trajectories.png


**Interpretation:**

Within the 20 longest-charting songs, Debut High is the most common trajectory (646 of the 1,294 classified weekly rows), which says that many of the biggest hits in 2023–2026 entered the chart strong rather than grinding up from the bottom. The three representative lines show that pattern clearly: Beautiful Things (Benson Boone) peaks fast and stays high (Debut High, 89 weeks), Pink Pony Club (Chappell Roan) builds gradually before exploding (Slow Climber, 68 weeks) — which matches the cultural “slow burn” narrative — and Lose Control (Teddy Swims) peaks early then carries a long tail (Spike & Drop, 112 weeks).

The honest surprise is the Lose Control label. Culturally it feels like a slow burn, but the rule `peak_position < 0.60` calls it Spike & Drop because its best rank lands in the first half of a 112-week run. This is a real limitation of fixed thresholds when total chart life ranges from a few weeks to over 100 — “peak position as a fraction of chart life” is sensitive to how long a song lasts, not just where its peak is. As a follow-up I would either retune the thresholds for very long-running songs, or switch to peak-rank timing in absolute weeks; I would also compare the trajectory mix for songs that never reached the Top 10 against those that did, to see whether climb-vs-debut behavior is mostly a story about hits.

**Question 3:** *Has chart behavior changed across 2023–2026 in terms of new entries, average 
   weeks on chart, and rank movement volatility?*

In [106]:
# Your analysis for Question 3
## Chart 3: Has Chart Behavior Changed Over Time? 
# Tracking new entries, average weeks on chart, and rank volatility  across 2023–2026 to see if chart behavior has shifted.

# --- Q3: AGGREGATE BY MONTH ---

# 1. New entries per month
new_entries = (df[df['is_new'] == True]
               .groupby(['year', 'month'])
               .size()
               .reset_index(name='new_entries'))

# 2. Average weeks on chart per month
avg_weeks = (df.groupby(['year', 'month'])['weeks_on_board']
             .mean()
             .reset_index(name='avg_weeks'))

# 3. Volatility: average absolute rank change per month
volatility = (df[df['rank_change'].notna()]
              .groupby(['year', 'month'])['rank_change']
              .apply(lambda x: x.abs().mean())
              .reset_index(name='avg_volatility'))

# Merge all three
from functools import reduce
monthly = reduce(lambda l, r: pd.merge(l, r, on=['year','month']),
                 [new_entries, avg_weeks, volatility])

# Create proper date column
monthly['date'] = pd.to_datetime(monthly[['year','month']].assign(day=1))
monthly = monthly.sort_values('date')

print(f"Total months: {len(monthly)}")
print(monthly.head(10).to_string())

Total months: 41
   year  month  new_entries  avg_weeks  avg_volatility       date
0  2023      1           22    13.7375        8.041916 2023-01-01
1  2023      2           36    13.5800        4.575843 2023-02-01
2  2023      3           61    12.6525       11.160606 2023-03-01
3  2023      4           46    13.3300        6.867117 2023-04-01
4  2023      5           28    14.4225        4.681199 2023-05-01
5  2023      6           46    13.5325        5.700581 2023-06-01
6  2023      7           94    12.7860        8.109415 2023-07-01
7  2023      8           55    11.9975        9.939940 2023-08-01
8  2023      9           66    11.3060       10.042453 2023-09-01
9  2023     10           69    10.8325       13.684375 2023-10-01


In [107]:
# Q3 — CHART: Has Chart Behavior Changed Over Time?

from plotly.subplots import make_subplots

# Remove incomplete last month
monthly_clean = monthly[monthly['date'] < '2026-05-01'].copy()

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=(
        'New Songs Entering the Chart Each Month',
        'Average Weeks on Chart Per Month',
        'Average Rank Change Volatility Per Month'
    ),
    vertical_spacing=0.08
)

# --- TRACE 1: New Entries ---
fig.add_trace(go.Scatter(
    x=monthly_clean['date'],
    y=monthly_clean['new_entries'],
    mode='lines+markers',
    name='New Entries',
    line=dict(color='#7b2d8b', width=2),
    marker=dict(size=5),
    hovertemplate='%{x|%b %Y}<br>New entries: %{y}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=monthly_clean['date'],
    y=monthly_clean['new_entries'].rolling(3, center=True).mean(),
    mode='lines',
    line=dict(color='#7b2d8b', width=1, dash='dash'),
    opacity=0.5,
    showlegend=False,
    hoverinfo='skip'
), row=1, col=1)

# --- TRACE 2: Avg Weeks on Chart ---
fig.add_trace(go.Scatter(
    x=monthly_clean['date'],
    y=monthly_clean['avg_weeks'].round(1),
    mode='lines+markers',
    name='Avg Weeks on Chart',
    line=dict(color='#e91e8c', width=2),
    marker=dict(size=5),
    hovertemplate='%{x|%b %Y}<br>Avg weeks: %{y}<extra></extra>'
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=monthly_clean['date'],
    y=monthly_clean['avg_weeks'].rolling(3, center=True).mean(),
    mode='lines',
    line=dict(color='#e91e8c', width=1, dash='dash'),
    opacity=0.5,
    showlegend=False,
    hoverinfo='skip'
), row=2, col=1)

# --- TRACE 3: Volatility ---
fig.add_trace(go.Scatter(
    x=monthly_clean['date'],
    y=monthly_clean['avg_volatility'].round(2),
    mode='lines+markers',
    name='Rank Volatility',
    line=dict(color='#f5a623', width=2),
    marker=dict(size=5),
    hovertemplate='%{x|%b %Y}<br>Volatility: %{y}<extra></extra>'
), row=3, col=1)

fig.add_trace(go.Scatter(
    x=monthly_clean['date'],
    y=monthly_clean['avg_volatility'].rolling(3, center=True).mean(),
    mode='lines',
    line=dict(color='#f5a623', width=1, dash='dash'),
    opacity=0.5,
    showlegend=False,
    hoverinfo='skip'
), row=3, col=1)

fig.update_layout(
    title=dict(
        text='<b>Billboard Hot 100</b><br>'
             '<sup>Has Chart Behavior Changed? (2023–2026)</sup>',
        font=dict(size=20, color='#2d1a2e'),
        x=0.5
    ),
    plot_bgcolor='#fdf0f5',
    paper_bgcolor='#f5eaf5',
    height=750,
    margin=dict(l=60, r=60, t=120, b=60),
    hovermode='x unified',
    showlegend=False,
    font=dict(color='#2d1a2e')
)

for i in range(1, 4):
    fig.update_xaxes(
        gridcolor='#e8c8e0',
        tickfont=dict(color='#666666'),
        row=i, col=1
    )
    fig.update_yaxes(
        gridcolor='#e8c8e0',
        tickfont=dict(color='#666666'),
        row=i, col=1
    )

fig.update_yaxes(title_text='New Songs', row=1, col=1)
fig.update_yaxes(title_text='Avg Weeks', row=2, col=1)
fig.update_yaxes(title_text='Volatility', row=3, col=1)

fig.show()
fig.write_image('chart3_chart_behavior.png', width=1400, height=800, scale=2)
print("Saved as chart3_chart_behavior.png")

Saved as chart3_chart_behavior.png


**Interpretation:**

The clearest takeaway across the three panels is that no single metric drifts steadily in one direction across 2023–2026 — the chart behavior is spiky and seasonal rather than a smooth structural shift. New entries swing month to month (e.g. a burst of 94 in July 2023 against months in the 20s and 30s), and the rolling-average overlay smooths the noise without revealing a clear “more new music every month” trend. Average weeks on chart dips in the middle of the window and recovers later, but `weeks_on_board` is Billboard’s cumulative chart history rather than weeks alive inside this dataset, so Panel 2 is a proxy for in-window longevity, not a pure measure of it. Volatility stays mostly in a band with occasional spikes (e.g. ~13.7 average absolute rank change in October 2023), so chart churn is real but not uniformly increasing.

I expected to see a clean upward trend in new entries or volatility — the “TikTok era, everything turns over faster” narrative — and that is not what the data shows for this window. A practitioner glancing at this should walk away thinking the Hot 100 in this period is shaped by bursts and events (releases, holidays, viral moments) more than by one macro trend, which is a useful, non-overclaimed finding on its own. As a follow-up I would split new entries from re-entries (744 re-entries in the movement breakdown, including holiday returners like All I Want For Christmas Is You) to see whether the type of turnover is changing even when the total count is not, and I would re-do the longevity panel using only weeks observed inside the 2023–2026 window to remove the cumulative-history confound.

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [108]:
# Section 4 — Visualization: Top-10 weeks vs weeks at #1
# Finding: many songs run forever in the Top 10 without ever reaching #1.

# --- BUILD DATA ---
presence = df.groupby(['song', 'primary_artist']).agg(
    top10_weeks   = ('is_top_10', 'sum'),
    number1_weeks = ('is_number_one', 'sum'),
    peak_rank     = ('rank', 'min'),
).reset_index()

# Keep songs with meaningful Top-10 presence so the scatter shows spread, not noise
presence = presence[presence['top10_weeks'] >= 5].copy()

presence['song_key'] = presence['song'] + ' — ' + presence['primary_artist']
presence['hover_tpl'] = presence.apply(
    lambda r: build_hover_text(
        r['song'], r['primary_artist'],
        [f"Peak Rank: #{r['peak_rank']}",
         f"Weeks in Top 10: {r['top10_weeks']}",
         f"Weeks at #1: {r['number1_weeks']}"],
    ), axis=1,
)

# Outliers to highlight (support the claim)
# Each: (song title in data, short label, annotation offset ax, ay)
highlights = [
    ('Lose Control',       'Lose Control — Teddy Swims',          40, -30),
    ('A Bar Song (Tipsy)', 'A Bar Song — Shaboozey',              40,  30),
    ('Espresso',           'Espresso — Sabrina Carpenter',       -90, -40),
    ('Birds Of A Feather', 'Birds Of A Feather — Billie Eilish',  60,  40),
]

highlight_titles = [h[0] for h in highlights]
is_highlight = presence['song'].isin(highlight_titles)
regular = presence[~is_highlight]
highlight = presence[is_highlight]

# --- BUILD FIGURE ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=regular['top10_weeks'],
    y=regular['number1_weeks'],
    mode='markers',
    marker=dict(
        color='#7b2d8b',
        size=10,
        opacity=0.65,
        line=dict(color='white', width=1),
    ),
    hovertemplate=regular['hover_tpl'],
    hoverlabel=HOVER_LABEL,
    name='All songs (≥5 Top-10 weeks)',
))

fig.add_trace(go.Scatter(
    x=highlight['top10_weeks'],
    y=highlight['number1_weeks'],
    mode='markers',
    marker=dict(
        color='#e91e8c',
        size=14,
        line=dict(color='white', width=2),
    ),
    hovertemplate=highlight['hover_tpl'],
    hoverlabel=HOVER_LABEL,
    name='Highlighted outliers',
))

for song_title, label, ax_off, ay_off in highlights:
    row = presence[presence['song'] == song_title].head(1)
    if row.empty:
        continue
    fig.add_annotation(
        x=float(row['top10_weeks'].iloc[0]),
        y=float(row['number1_weeks'].iloc[0]),
        text=label,
        showarrow=True,
        arrowhead=2,
        arrowcolor='#2d1a2e',
        ax=ax_off, ay=ay_off,
        font=dict(size=11, color='#2d1a2e'),
        bgcolor='rgba(253,240,245,0.9)',
        bordercolor='#e8c8e0',
        borderwidth=1,
    )

fig.update_layout(
    title=dict(
        text="<b>Long Top-10 runs don't guarantee weeks at #1</b><br>"
             '<sup>Billboard Hot 100 songs, 2023–2026</sup>',
        font=dict(size=20, color='#2d1a2e'),
        x=0.5,
    ),
    xaxis=dict(
        title='Weeks in Top 10',
        gridcolor='#e8c8e0',
        tickfont=dict(color='#666666'),
        zeroline=False,
    ),
    yaxis=dict(
        title='Weeks at #1',
        gridcolor='#e8c8e0',
        tickfont=dict(color='#666666'),
        zeroline=True,
        zerolinecolor='#aaaaaa',
    ),
    plot_bgcolor='#fdf0f5',
    paper_bgcolor='#f5eaf5',
    height=600,
    margin=dict(l=80, r=80, t=100, b=80),
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.15,
        xanchor='center',
        x=0.5,
        font=dict(size=12, color='#2d1a2e'),
        bgcolor='rgba(253,240,245,0.8)',
        bordercolor='#e8c8e0',
        borderwidth=1,
    ),
)

fig.show()
fig.write_image('chart4_top10_vs_number1.png', width=1400, height=700, scale=2)
print("Saved as chart4_top10_vs_number1.png")

Saved as chart4_top10_vs_number1.png


**Chart rationale:**

I chose a scatter plot because the finding is about the *relationship* between two numeric outcomes per song — weeks in the Top 10 and weeks at #1 — and a scatter is the only chart type where decoupling between two metrics shows up directly as position. Chart 1 in Section 3 already ranks songs on a single axis with stacked bars, so reusing a bar here would just restate the same view. By contrast, the scatter makes the gap visible: most points sit at or near `y = 0` even when `x` is large, which is the literal shape of “a song can live in the Top 10 for months without ever spending a week at #1.” I annotated four songs that anchor the story — Lose Control (80 Top-10 weeks, 1 at #1), A Bar Song (Tipsy) (66, 19), Espresso (33, 0), and Birds Of A Feather (33, 0) — so the reader can find the named outliers without hunting through hover text.

The takeaway I want a practitioner (label A&R, playlist editor, music supervisor) to leave with is that “stayed in the Top 10 forever” and “dominated #1” are two different success stories and should not be collapsed into one popularity number. If you are choosing songs to license, market, or build a campaign around, you need to look at both axes — a song like Lose Control is a long-tail attention play, while A Bar Song (Tipsy) is a peak-dominance play, and treating either as the “real” signal of success would miss the other half of the chart.

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**

The most important thing this analysis revealed is that **longevity and dominance on the Hot 100 are two different stories** — Lose Control (Teddy Swims) held 80 weeks in the Top 10 with only 1 week at #1, while A Bar Song (Tipsy) (Shaboozey) spent 19 weeks at #1 from a shorter 66-week Top-10 run, and a “popularity” number that collapses these into one would hide the gap. What surprised me was that **Chart 3 did not show a clean structural shift** across 2023–2026: new entries, average longevity, and rank volatility are spiky and seasonal rather than trending in one direction, so the “TikTok era, everything turns over faster” narrative I expected to confirm is not visible in this window. If I had more time I would do an **artist-level rollup** (total weeks and #1 count per `primary_artist`) and **re-compute longevity using only weeks observed inside 2023–2026** to remove the cumulative `weeks_on_board` confound that biases Chart 3’s middle panel. The main limitations are that `primary_artist` collapses collaborations to the first-billed name (so true duets like Lady Gaga & Bruno Mars get under-credited), the Q2 trajectory categories use fixed thresholds that break down for very long chart runs (Lose Control reads as Spike & Drop despite feeling culturally like a slow burn), and the dataset is **a popularity proxy only** — it does not carry streams, sales, genre, demographics, or any signal about *why* a song charted.

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.

## Project Wrap-Up

**What this project is.** A pandas + Plotly analysis of weekly Billboard Hot 100 data from 2023-01-07 to 2026-05-02 — 17,400 rows across 174 chart weeks, covering 2,181 unique songs and 506 unique primary artists — built to answer three questions about chart presence, song trajectories, and how chart behavior has shifted over time.

**Four charts, each tied to a finding.**

- *Chart 1 — Top 10 Most Charted Songs* (horizontal bar): ranks songs by weeks in the Top 10, with weeks at #1 overlaid on the same row so the longevity-vs-dominance gap is visible at a glance.
- *Chart 2 — Three Ways a Song Can Win* (multi-line): one representative song per algorithmic trajectory category — Beautiful Things (Debut High, 89 weeks), Pink Pony Club (Slow Climber, 68 weeks), Lose Control (Spike & Drop, 112 weeks).
- *Chart 3 — Has Chart Behavior Changed?* (three stacked line panels with rolling-average overlays): new entries, average weeks on chart, and rank volatility tracked monthly across 41 months.
- *Chart 4 — Long Top-10 runs don't guarantee weeks at #1* (Section 4 scatter): the same Q1 finding viewed as a relationship between two metrics, with four annotated outliers (Lose Control, A Bar Song (Tipsy), Espresso, Birds Of A Feather).

**Numbers worth remembering.**

- Lose Control: 80 Top-10 weeks, 1 week at #1 — the clearest longevity story.
- A Bar Song (Tipsy): 19 weeks at #1 — the clearest dominance story.
- Espresso and Birds Of A Feather sit at exactly the same point — 33 Top-10 weeks, 0 weeks at #1 — which is the literal shape of “popular is not the same as #1.”
- Of 17,400 weekly rows, only ~12% (2,044) are new entries; 744 are re-entries, large enough to materially shape longevity stats (e.g. holiday returners like All I Want For Christmas Is You).

**What I would not claim from this dataset.** Causation (no listener, streaming, or marketing data to explain *why* a song charted); genre-level conclusions (no genre column in the source); or an “era is changing” claim — Chart 3 shows seasonal spikes and event-driven bursts, not a clean structural trend across 2023–2026.

**What's next.** Artist-level rollups (count of distinct charting songs plus total weeks per `primary_artist`), an in-window-only longevity recomputation that ignores Billboard’s cumulative `weeks_on_board`, and trajectory classification tuned for songs whose chart life runs far past the median — so categories like Spike & Drop and Slow Climber stop being thrown off by very long runs.